# Phase 4: Bayesian Sentinel-2 inversion and spatial mapping

This notebook runs Phase 4 on a CPU runtime:
- Synthetic-truth experiment: about 2–4 minutes.
- Real Sentinel-2 Dark Zone scene: about 5–8 minutes.

Imagery is streamed from the public AWS archive, so no account is needed.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "pyproj", "pystac-client", "cartopy",
                "emcee", "miepython", "pyyaml", "seaborn", "tqdm"], check=True)
!git clone -q -b claude/sweet-noether-6pryo4 https://github.com/avinautiya/algae.git || true
!git clone -q --depth 1 https://github.com/jmcook1186/biosnicar-py.git || true
os.environ["BIOSNICAR_PATH"] = "/content/biosnicar-py"
%cd /content/algae
!python phase4/tests/test_phase4.py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
P1 = '/content/drive/MyDrive/algae_phase1'
OUT = '/content/drive/MyDrive/algae_phase4'

In [ ]:
# 1) Synthetic-truth experiment (error reduction vs Tier A and the empirical index), and the reverse test
!python phase4/run_phase4.py --source synthetic --phase1-l2 {P1}/level2 --outdir {OUT}/synthetic --workers 2
!python phase4/run_phase4.py --source synthetic --truth-model tierA --phase1-l2 {P1}/level2 --outdir {OUT}/synthetic_reverse --workers 2

In [ ]:
# 2) Real Sentinel-2 L2A scene, SW Greenland Dark Zone (23 Jul 2019). Field validation on 31 co-located samples runs first; add --phenol williamson2020 for the measured phenolic MAC.
!python phase4/run_phase4.py --source s2 --scene-id S2A_22WEV_20190723_0_L2A --phase1-l2 {P1}/level2 \
    --resolution 40 --f-step 0.2 --outdir {OUT}/real --workers 2

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(f"{OUT}/real/tables/field_validation_metrics.csv").round(3))  # 31 field samples (Cook et al. 2020)
display(pd.read_csv(f"{OUT}/synthetic/tables/validation_metrics.csv").round(3))
display(pd.read_csv(f"{OUT}/synthetic/tables/error_reduction_vs_tierA.csv").round(1))
for run in ("synthetic", "real"):
    for f in ("Fig4A_biomass_maps", "Fig4B_albedo_forcing_maps", "Fig4C_corner_pixel1"):
        display(Image(f"{OUT}/{run}/figures/{f}.png", width=950))